In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

pal = ['#4E79A7','#F28E2B','#E15759','#76B7B2','#59A14F','#EDC948']
RANDOM_STATE = 42

In [ ]:
# Ucitavanje podataka

df_train = pd.read_csv("data/train.csv")
df_test  = pd.read_csv("data/test.csv")

feature_cols = [c for c in df_train.columns
                if c not in ("Activity", "ActivityName", "subject")]

X_train = df_train[feature_cols].values
y_train = df_train["ActivityName"].values
X_test  = df_test[feature_cols].values
y_test  = df_test["ActivityName"].values

# Skaliranje
scaler   = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
# Pomocna funkcija za evaluaciju

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"  {name}")
    print(f"  Tacnost: {acc:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return acc

rezultati = {}

In [ ]:
# 1. Baseline — majority class

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
rezultati["Majority class"] = evaluate("Majority class (baseline)", y_test, y_pred_dummy)

In [ ]:
# 2. Baseline — (staticne vs dinamicne aktivnosti)
from sklearn.tree import DecisionTreeClassifier, export_text

staticne  = {"LAYING", "SITTING", "STANDING"}
dinamicne = {"WALKING", "WALKING_DOWNSTAIRS", "WALKING_UPSTAIRS"}

y_train_bin = np.where(np.isin(y_train, list(staticne)), "staticna", "dinamicna")
y_test_bin  = np.where(np.isin(y_test,  list(staticne)), "staticna", "dinamicna")

stump_bin = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump_bin.fit(X_train, y_train_bin)

print("Korisceni prediktor (feature index):", stump_bin.tree_.feature[0])
print("Feature naziv:", feature_cols[stump_bin.tree_.feature[0]])
print(export_text(stump_bin, feature_names=feature_cols))

rezultati["Decision stump (bin)"] = evaluate("Decision stump (staticno/dinamicno)", y_test_bin, stump_bin.predict(X_test))

In [ ]:
# 3. Logisticka regresija

model_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_lr.fit(X_train_sc, y_train)
y_pred_lr = model_lr.predict(X_test_sc)

rezultati["Logisticka regresija"] = evaluate("Logisticka regresija", y_test, y_pred_lr)

In [ ]:

# 4. SVM sa RBF kernelom — GridSearch 10-fold CV

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  SVC(kernel="rbf", random_state=RANDOM_STATE))
])

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

param_grid_svm = {
    "model__C"    : [1, 10, 100, 1000],
    "model__gamma": ["scale", 0.001, 0.01, 0.1],
}

gs_svm = GridSearchCV(
    pipe_svm,
    param_grid_svm,
    cv=cv10,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train, y_train)

print(f"Najbolji parametri: {gs_svm.best_params_}")
print(f"Najbolja CV tacnost: {gs_svm.best_score_:.4f}")

y_pred_svm = gs_svm.predict(X_test)
rezultati["SVM (RBF)"] = evaluate("SVM sa RBF kernelom", y_test, y_pred_svm)

In [ ]:
# 5. Random Forest — GridSearch 5-fold CV


param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth"   : [None, 20, 40],
    "min_samples_split": [2, 5],
}

print("Random Forest GridSearch...")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf,
    scoring="accuracy",
    n_jobs=-1,
)
gs_rf.fit(X_train, y_train)

print(f"Najbolji parametri: {gs_rf.best_params_}")
print(f"Najbolja CV tacnost: {gs_rf.best_score_:.4f}")

y_pred_rf = gs_rf.predict(X_test)
rezultati["Random Forest"] = evaluate("Random Forest", y_test, y_pred_rf)

In [ ]:
# 6. Gradient Boosting — GridSearch 5-fold CV


param_grid_gbt = {
    "n_estimators"  : [100, 200],
    "learning_rate" : [0.05, 0.1],
    "max_depth"     : [3, 5],
}

print("Gradient Boosting GridSearch — ovo ce potrajati duze...")
gs_gbt = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid_gbt,
    scoring="accuracy",
    n_jobs=-1,

)
gs_gbt.fit(X_train, y_train)

print(f"Najbolji parametri: {gs_gbt.best_params_}")
print(f"Najbolja CV tacnost: {gs_gbt.best_score_:.4f}")

y_pred_gbt = gs_gbt.predict(X_test)
rezultati["Gradient Boosting"] = evaluate("Gradient Boosting", y_test, y_pred_gbt)

In [ ]:

# 7. Tabela poredenja modela


df_rezultati = pd.DataFrame(
    list(rezultati.items()),
    columns=["Model", "Tacnost (test)"]
).sort_values("Tacnost (test)", ascending=False)

print("\n" + "="*45)
print(df_rezultati.to_string(index=False))
print("="*45)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_rezultati["Model"], df_rezultati["Tacnost (test)"],
               color=pal[:len(df_rezultati)], edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
ax.set_xlabel("Tacnost na test skupu")
ax.set_title("Poredenje modela — UCI HAR Dataset", fontweight="bold")
ax.set_xlim(0, 1.05)
sns.despine()
plt.tight_layout()
plt.savefig("poredenje_modela.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# 8. Konfuziona matrica — SVM (referentni model iz literature)

labels = sorted(df_test["ActivityName"].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (naziv, y_pred) in zip(axes, [
    ("Logisticka regresija", y_pred_lr),
    ("SVM (RBF)",            y_pred_svm),
]):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Konfuziona matrica — {naziv}", fontweight="bold")
    ax.set_ylabel("Stvarna klasa")
    ax.set_xlabel("Predvidjena klasa")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("konfuziona_matrica_lr_svm.png", dpi=120, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (naziv, y_pred) in zip(axes, [
    ("Random Forest",    y_pred_rf),
    ("Gradient Boosting", y_pred_gbt),
]):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Konfuziona matrica — {naziv}", fontweight="bold")
    ax.set_ylabel("Stvarna klasa")
    ax.set_xlabel("Predvidjena klasa")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("konfuziona_matrica_rf_gbt.png", dpi=120, bbox_inches="tight")
plt.show()